### Proyecto 6 (v1): predictor de precio desde descripcion de producto

Objetivo: dado el texto de una descripcion de producto, predecir el precio. Tres etapas del modulo, en orden:

1. **Curacion de datos** -- el dataset sintetico viene con problemas a proposito (nulos, duplicados, texto sucio) para practicar limpiarlo antes de usarlo.
2. **Feature engineering** -- convertir el texto en numeros que un modelo clasico de ML pueda usar.
3. **Linea base con ML clasico** (regresion lineal / random forest) vs **LLM directo** (Gemini) -- comparar el error de cada enfoque. Sin esta comparacion no sabriamos si vale la pena la complejidad de usar un LLM.

#### Paso 0: generar el dataset sintetico

200 productos ficticios en 5 categorias, con un precio real generado por formula (precio base de categoria x multiplicador por adjetivos tipo "premium"/"basico" + ruido aleatorio). A proposito metemos filas sucias: nulos, precios duplicados exactos, espacios de mas -- para tener algo real que curar en el paso 1.

In [7]:
import random
import pandas as pd

random.seed(42)

categorias = {
    "electronica": {"base": 25000, "items": ["auriculares bluetooth", "mouse inalambrico", "teclado mecanico", "parlante portatil", "cargador rapido"]},
    "hogar": {"base": 8000, "items": ["set de sabanas", "lampara de mesa", "juego de ollas", "cortina blackout", "organizador de closet"]},
    "ropa": {"base": 6000, "items": ["campera de abrigo", "zapatillas urbanas", "remera de algodon", "jean clasico", "buzo con capucha"]},
    "deportes": {"base": 12000, "items": ["pelota de futbol", "bicicleta rodado 26", "mancuernas 5kg", "colchoneta de yoga", "mochila de trekking"]},
    "libros": {"base": 3000, "items": ["novela de misterio", "manual de programacion", "libro de cocina", "biografia", "comic ilustrado"]},
}

adjetivos_premium = ["edicion premium", "calidad superior", "gama alta", "importado"]
adjetivos_basico = ["version basica", "economico", "entrada de gama"]

filas = []
for _ in range(200):
    categoria = random.choice(list(categorias.keys()))
    info = categorias[categoria]
    item = random.choice(info["items"])

    tipo = random.choices(["premium", "basico", "normal"], weights=[0.25, 0.25, 0.5])[0]
    if tipo == "premium":
        adjetivo = random.choice(adjetivos_premium)
        multiplicador = random.uniform(1.6, 2.3)
    elif tipo == "basico":
        adjetivo = random.choice(adjetivos_basico)
        multiplicador = random.uniform(0.5, 0.75)
    else:
        adjetivo = ""
        multiplicador = random.uniform(0.85, 1.25)

    descripcion = f"{item} {adjetivo}".strip()
    precio = round(info["base"] * multiplicador + random.uniform(-500, 500), -1)

    filas.append({"descripcion": descripcion, "categoria": categoria, "precio": precio})

df = pd.DataFrame(filas)

# ensuciar el dataset a proposito
df.loc[5:8, "precio"] = None
df = pd.concat([df, df.loc[[15, 40, 90]]], ignore_index=True)  # duplicados exactos
df.loc[20, "descripcion"] = "   " + df.loc[20, "descripcion"].upper() + "   "

print(f"dataset crudo: {len(df)} filas")
df.head(20)

dataset crudo: 203 filas


,descripcion,categoria,precio
0,auriculares bluetooth,electronica,23340.0
1,cargador rapido importado,electronica,40150.0
2,organizador de closet,hogar,8810.0
3,biografia gama alta,libros,6010.0
4,cortina blackout version basica,hogar,4690.0
5,auriculares bluetooth economico,electronica,NaN
6,parlante portatil,electronica,NaN
7,libro de cocina,libros,NaN
8,manual de programacion,libros,NaN
9,campera de abrigo,ropa,7060.0


#### Paso 1: curacion de datos

Sacamos: filas con precio nulo (no sirven para entrenar ni evaluar), duplicados exactos (inflarian artificialmente la importancia de esos productos), y normalizamos texto (trim + minusculas) para que "REMERA" y "remera" no se traten como cosas distintas en el feature engineering.

In [2]:
print(f"nulos en precio: {df['precio'].isna().sum()}")
print(f"duplicados exactos: {df.duplicated().sum()}")

df_limpio = df.dropna(subset=["precio"]).drop_duplicates().copy()
df_limpio["descripcion"] = df_limpio["descripcion"].str.strip().str.lower()
df_limpio = df_limpio.reset_index(drop=True)

print(f"dataset limpio: {len(df_limpio)} filas (de {len(df)} originales)")

nulos en precio: 4
duplicados exactos: 3
dataset limpio: 196 filas (de 203 originales)


#### Paso 2: feature engineering

Un modelo de regresion clasico no entiende texto -- necesita numeros. Extraemos features simples de la descripcion: cantidad de palabras, si menciona terminos de "premium", si menciona terminos de "basico", y la categoria (one-hot, porque es la senal mas fuerte de precio base).

In [3]:
terminos_premium = ["premium", "superior", "gama alta", "importado"]
terminos_basico = ["basica", "economico", "entrada de gama"]

df_limpio["cant_palabras"] = df_limpio["descripcion"].str.split().str.len()
df_limpio["es_premium"] = df_limpio["descripcion"].apply(lambda d: any(t in d for t in terminos_premium)).astype(int)
df_limpio["es_basico"] = df_limpio["descripcion"].apply(lambda d: any(t in d for t in terminos_basico)).astype(int)

features = pd.get_dummies(df_limpio[["cant_palabras", "es_premium", "es_basico", "categoria"]], columns=["categoria"])
print(features.columns.tolist())
features.head()

['cant_palabras', 'es_premium', 'es_basico', 'categoria_deportes', 'categoria_electronica', 'categoria_hogar', 'categoria_libros', 'categoria_ropa']


,cant_palabras,es_premium,es_basico,categoria_deportes,categoria_electronica,categoria_hogar,categoria_libros,categoria_ropa
0,2,0,0,False,True,False,False,False
1,3,1,0,False,True,False,False,False
2,3,0,0,False,False,True,False,False
3,3,1,0,False,False,False,True,False
4,4,0,1,False,False,True,False,False


#### Paso 3a: linea base con ML clasico

Random Forest sobre las features de arriba. Separamos 80/20 train/test y medimos el error promedio absoluto (MAE) en pesos -- cuanto se equivoca el modelo en promedio, en la misma unidad que el precio.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X = features
y = df_limpio["precio"]

X_train, X_test, y_train, y_test, desc_train, desc_test = train_test_split(
    X, y, df_limpio["descripcion"], test_size=0.2, random_state=42
)

modelo_clasico = RandomForestRegressor(n_estimators=200, random_state=42)
modelo_clasico.fit(X_train, y_train)
pred_clasico = modelo_clasico.predict(X_test)

mae_clasico = mean_absolute_error(y_test, pred_clasico)
print(f"MAE Random Forest: ${mae_clasico:,.0f}")

MAE Random Forest: $934


#### Paso 3b: linea base con LLM directo

Le pasamos a Gemini SOLO el texto de la descripcion (sin las features numericas que le dimos al Random Forest) y le pedimos que estime el precio. Es una comparacion injusta a proposito en un sentido -- el LLM no vio ejemplos de precios reales de este dataset -- pero es justo la pregunta que importa: ¿el conocimiento general del LLM sobre precios de productos le alcanza para competir con un modelo entrenado especificamente en estos datos?

In [5]:
from dotenv import load_dotenv
from google import genai
import os
import re
import time

load_dotenv()
cliente = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def estimar_precio_llm(descripcion):
    prompt = f"""Estima el precio en pesos argentinos de este producto, basandote en precios tipicos de mercado.

Producto: "{descripcion}"

Respondé SOLO con el numero, sin simbolo de moneda ni texto."""
    respuesta = cliente.models.generate_content(model="gemini-flash-lite-latest", contents=prompt)
    numeros = re.findall(r"[\d.]+", respuesta.text.replace(",", ""))
    return float(numeros[0]) if numeros else None

# free tier: 15 requests/minuto compartido a nivel cuenta -- espaciamos 6.5s y limitamos la muestra
time.sleep(15)
muestra_test = desc_test.head(10)
pred_llm = []
for d in muestra_test:
    pred_llm.append(estimar_precio_llm(d))
    time.sleep(6.5)

y_test_muestra = y_test.loc[muestra_test.index]

mae_llm = mean_absolute_error(y_test_muestra, pred_llm)
print(f"MAE LLM (Gemini, muestra de {len(muestra_test)}): ${mae_llm:,.0f}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


MAE LLM (Gemini, muestra de 10): $50,099


#### Comparacion final

In [6]:
mae_clasico_muestra = mean_absolute_error(y_test_muestra, modelo_clasico.predict(X_test.loc[muestra_test.index]))

print(f"Sobre la misma muestra de {len(muestra_test)} productos:")
print(f"  Random Forest (features numericas): ${mae_clasico_muestra:,.0f} de error promedio")
print(f"  Gemini (solo texto, sin entrenar):  ${mae_llm:,.0f} de error promedio")

Sobre la misma muestra de 10 productos:
  Random Forest (features numericas): $674 de error promedio
  Gemini (solo texto, sin entrenar):  $50,099 de error promedio


**Nota sobre fine-tuning (queda para proyecto 7):** el siguiente paso natural seria ajustar un modelo (fine-tuning de OpenAI, o QLoRA propio) con ejemplos de ESTE dataset especifico, para ver si un LLM entrenado en estos datos le gana al Random Forest. Eso es exactamente lo que hace el proyecto 7 del modulo 7 -- esta v1 deja la linea base para comparar contra eso.